## 1. Setup


In [1]:
!pip install -q rasterio scikit-learn

from pathlib import Path
import json
import warnings
import logging
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import tensorflow as tf

# Rasterio can emit non-fatal TIFF metadata warnings for Earth Engine exports.
logging.getLogger("rasterio").setLevel(logging.ERROR)

from google.colab import drive
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


drive.mount("/content/drive")

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Lucy_CNN"
)

UNSUPERVISED_DIR = (
    PROJECT_DIR / "unsupervised_cnn"
)

UNSUPERVISED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MANIFEST_PATH = (
    UNSUPERVISED_DIR / "unsupervised_manifest.csv"
)

CONFIG_PATH = (
    UNSUPERVISED_DIR / "preprocessing_config.json"
)

TUNING_RESULTS_PATH = (
    UNSUPERVISED_DIR / "hyperparameter_tuning.csv"
)

BEST_PARAMS_PATH = (
    UNSUPERVISED_DIR / "best_autoencoder_parameters.json"
)

MODEL_PATH = (
    UNSUPERVISED_DIR / "normal_port_autoencoder.keras"
)

ENCODER_PATH = (
    UNSUPERVISED_DIR / "normal_port_encoder.keras"
)

SCALING_PATH = (
    UNSUPERVISED_DIR / "sar_scaling.json"
)

PATCH_RESULTS_PATH = (
    UNSUPERVISED_DIR / "unsupervised_patch_results.csv"
)

PORT_RESULTS_PATH = (
    UNSUPERVISED_DIR / "unsupervised_port_results.csv"
)

CLUSTER_PATH = (
    UNSUPERVISED_DIR / "kmeans_anomaly_model.json"
)

MAX_TUNING_TRIALS = 12
TUNING_EPOCHS = 12
FINAL_EPOCHS = 40

print("Outputs:", UNSUPERVISED_DIR)


Mounted at /content/drive
Outputs: /content/drive/MyDrive/Lucy_CNN/unsupervised_cnn


## 2. Read the exported manifest

The manifest is produced by the final cell of the data-preparation notebook. This keeps data acquisition/preprocessing separate from the CNN stage.


In [2]:
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "Run the export cell at the end of the data-preparation "
        f"notebook first. Missing: {MANIFEST_PATH}"
    )

if not CONFIG_PATH.exists():
    raise FileNotFoundError(
        f"Missing preprocessing configuration: {CONFIG_PATH}"
    )

manifest = pd.read_csv(
    MANIFEST_PATH
)

with open(CONFIG_PATH) as file:
    preprocessing_config = json.load(file)

PATCH_DIR = (
    PROJECT_DIR / "Sentinel1_grid_patches"
)

# Use the project folder as the source of truth so an old config file
# cannot point the CNN to a previous Drive location.
if not PATCH_DIR.exists():
    raise FileNotFoundError(
        f"Patch directory not found: {PATCH_DIR}. "
        "Run the updated data-preparation notebook first."
    )

BAND_NAMES = list(
    preprocessing_config["band_names"]
)

NODATA_VALUE = float(
    preprocessing_config["nodata_value"]
)

PATCH_HEIGHT = int(
    preprocessing_config["patch_size_pixels"]
)

PATCH_WIDTH = PATCH_HEIGHT

EXPECTED_BANDS = 6

manifest["raster_path"] = manifest[
    "image_file"
].apply(
    lambda filename: str(PATCH_DIR / filename)
)

missing_files = [
    path
    for path in manifest["raster_path"]
    if not Path(path).exists()
]

if missing_files:
    raise FileNotFoundError(
        f"{len(missing_files)} patch files are missing from {PATCH_DIR}. "
        f"First missing file: {missing_files[0]}"
    )

print("Patches:", len(manifest))
print("Port-events:", manifest["source_port_id"].nunique())
print("Patch directory:", PATCH_DIR)

display(manifest.head())


Patches: 36
Port-events: 4
Patch directory: /content/drive/MyDrive/Lucy_CNN/Sentinel1_grid_patches


,sample_id,source_port_id,port_key,sample_type,port_name,country,world_port_index,latitude,longitude,north_offset_m,...,post_end,orbit_pass,relative_orbit,water_occurrence_threshold,nodata_value,export_crs,number_pre_images,number_post_images,image_file,raster_path
0,luganville_vu_harold_2020_r1_c1,luganville_vu_harold_2020,luganville_vu,event,LUGANVILLE,VU,57200,-15.521039,167.182063,-640,...,2020-04-17,DESCENDING,59,50,-9999.0,EPSG:32758,1,1,luganville_vu_harold_2020_r1_c1.tif,/content/drive/MyDrive/Lucy_CNN/Sentinel1_grid...
1,luganville_vu_harold_2020_r1_c2,luganville_vu_harold_2020,luganville_vu,event,LUGANVILLE,VU,57200,-15.521039,167.188030,-640,...,2020-04-17,DESCENDING,59,50,-9999.0,EPSG:32758,1,1,luganville_vu_harold_2020_r1_c2.tif,/content/drive/MyDrive/Lucy_CNN/Sentinel1_grid...
2,luganville_vu_harold_2020_r1_c3,luganville_vu_harold_2020,luganville_vu,event,LUGANVILLE,VU,57200,-15.521039,167.193997,-640,...,2020-04-17,DESCENDING,59,50,-9999.0,EPSG:32758,1,1,luganville_vu_harold_2020_r1_c3.tif,/content/drive/MyDrive/Lucy_CNN/Sentinel1_grid...
3,luganville_vu_harold_2020_r2_c1,luganville_vu_harold_2020,luganville_vu,event,LUGANVILLE,VU,57200,-15.515290,167.182063,0,...,2020-04-17,DESCENDING,59,50,-9999.0,EPSG:32758,1,1,luganville_vu_harold_2020_r2_c1.tif,/content/drive/MyDrive/Lucy_CNN/Sentinel1_grid...
4,luganville_vu_harold_2020_r2_c2,luganville_vu_harold_2020,luganville_vu,event,LUGANVILLE,VU,57200,-15.515290,167.188030,0,...,2020-04-17,DESCENDING,59,50,-9999.0,EPSG:32758,1,1,luganville_vu_harold_2020_r2_c2.tif,/content/drive/MyDrive/Lucy_CNN/Sentinel1_grid...


## 3. Validate the six-band GeoTIFFs


In [3]:
input_checks = []

for raster_path in manifest["raster_path"]:
    with rasterio.open(raster_path) as src:
        if src.count != EXPECTED_BANDS:
            raise ValueError(
                f"{raster_path} contains {src.count} bands; "
                f"expected {EXPECTED_BANDS}."
            )

        if (src.height, src.width) != (
            PATCH_HEIGHT,
            PATCH_WIDTH,
        ):
            raise ValueError(
                f"Unexpected raster size for {raster_path}: "
                f"{src.height} x {src.width}."
            )

        descriptions = list(src.descriptions)

        input_checks.append(
            {
                "file": Path(raster_path).name,
                "height": src.height,
                "width": src.width,
                "bands": src.count,
                "nodata": src.nodata,
                "band_descriptions": descriptions,
            }
        )

checks_df = pd.DataFrame(input_checks)
display(checks_df.head())

print("All patch files passed the shape and band-count checks.")


,file,height,width,bands,nodata,band_descriptions
0,luganville_vu_harold_2020_r1_c1.tif,128,128,6,-9999.0,"[pre_VV, pre_VH, post_VV, post_VH, change_VV, ..."
1,luganville_vu_harold_2020_r1_c2.tif,128,128,6,-9999.0,"[pre_VV, pre_VH, post_VV, post_VH, change_VV, ..."
2,luganville_vu_harold_2020_r1_c3.tif,128,128,6,-9999.0,"[pre_VV, pre_VH, post_VV, post_VH, change_VV, ..."
3,luganville_vu_harold_2020_r2_c1.tif,128,128,6,-9999.0,"[pre_VV, pre_VH, post_VV, post_VH, change_VV, ..."
4,luganville_vu_harold_2020_r2_c2.tif,128,128,6,-9999.0,"[pre_VV, pre_VH, post_VV, post_VH, change_VV, ..."


All patch files passed the shape and band-count checks.


## 4. Split by physical port

Event and control samples from the same physical port are kept in the same split. This prevents the CNN from seeing the same port geometry in training and then being evaluated on a different date from that same port.


In [4]:
# Split by physical port so event and control samples from the same location stay together.
required_split_columns = {
    "port_key",
    "sample_type",
    "source_port_id",
}

missing_split_columns = (
    required_split_columns
    - set(manifest.columns)
)

if missing_split_columns:
    raise ValueError(
        "The manifest is missing the new storm/control fields: "
        f"{sorted(missing_split_columns)}. "
        "Rerun the export cell in the updated data-preparation notebook."
    )

port_groups = np.array(
    sorted(manifest["port_key"].dropna().unique())
)

if len(port_groups) < 3:
    raise ValueError(
        "At least three physical ports are required to create "
        "train, validation and test groups."
    )

rng = np.random.default_rng(SEED)
shuffled_ports = port_groups.copy()
rng.shuffle(shuffled_ports)

n_ports = len(shuffled_ports)

n_test = max(1, int(round(0.15 * n_ports)))
n_val = max(1, int(round(0.15 * n_ports)))

if n_test + n_val >= n_ports:
    n_test = 1
    n_val = 1

n_train = n_ports - n_val - n_test

train_ports = set(shuffled_ports[:n_train])
val_ports = set(
    shuffled_ports[n_train:n_train + n_val]
)
test_ports = set(
    shuffled_ports[n_train + n_val:]
)

train_df = manifest[
    manifest["port_key"].isin(train_ports)
].reset_index(drop=True)

val_df = manifest[
    manifest["port_key"].isin(val_ports)
].reset_index(drop=True)

test_df = manifest[
    manifest["port_key"].isin(test_ports)
].reset_index(drop=True)

assert train_ports.isdisjoint(val_ports)
assert train_ports.isdisjoint(test_ports)
assert val_ports.isdisjoint(test_ports)

for split_name, frame, groups in [
    ("Training", train_df, train_ports),
    ("Validation", val_df, val_ports),
    ("Test", test_df, test_ports),
]:
    print(split_name)
    print("Physical ports:", len(groups))
    print("Patches:", len(frame))
    print("Port keys:", sorted(groups))
    print("Sample types:")
    print(
        frame[["source_port_id", "sample_type"]]
        .drop_duplicates()["sample_type"]
        .value_counts()
    )
    print()


Training
Physical ports: 1
Patches: 18
Port keys: [np.str_('suva_harbor_fj')]
Sample types:
sample_type
event               1
control_temporal    1
Name: count, dtype: int64

Validation
Physical ports: 1
Patches: 9
Port keys: [np.str_('saipan_mp')]
Sample types:
sample_type
control_temporal    1
Name: count, dtype: int64

Test
Physical ports: 1
Patches: 9
Port keys: [np.str_('luganville_vu')]
Sample types:
sample_type
event    1
Name: count, dtype: int64



## 5. Scale VV and VH using pre-disaster training data only

The autoencoder learns normal pre-disaster port appearance. Scaling is therefore fitted only to the two pre-event channels from the training ports and then applied unchanged to both pre- and post-event images.

This preserves pre/post differences and avoids information from held-out ports influencing preprocessing.


In [5]:
def read_raw_patch(raster_path):
    with rasterio.open(raster_path) as src:
        raster = src.read().astype(np.float32)
        nodata = src.nodata

    valid = np.isfinite(raster)

    if nodata is not None:
        valid &= raster != nodata

    valid &= raster != NODATA_VALUE

    return raster, valid


training_values = [[], []]

for raster_path in train_df["raster_path"]:
    raster, valid = read_raw_patch(raster_path)

    for band_index in [0, 1]:
        band_values = raster[band_index][
            valid[band_index]
        ]

        if band_values.size:
            training_values[band_index].append(
                band_values
            )

sar_low = np.zeros(2, dtype=np.float32)
sar_high = np.ones(2, dtype=np.float32)

for band_index in [0, 1]:
    values = np.concatenate(
        training_values[band_index]
    )

    sar_low[band_index] = np.percentile(
        values,
        2,
    )

    sar_high[band_index] = np.percentile(
        values,
        98,
    )

    if sar_high[band_index] <= sar_low[band_index]:
        sar_high[band_index] = sar_low[band_index] + 1.0

scaling_df = pd.DataFrame(
    {
        "band": ["VV", "VH"],
        "p02": sar_low,
        "p98": sar_high,
    }
)

display(scaling_df)

with open(SCALING_PATH, "w") as file:
    json.dump(
        {
            "VV_low": float(sar_low[0]),
            "VV_high": float(sar_high[0]),
            "VH_low": float(sar_low[1]),
            "VH_high": float(sar_high[1]),
        },
        file,
        indent=2,
    )


,band,p02,p98
0,VV,-14.766017,5.866412
1,VH,-21.955759,-1.795076


In [6]:
def scale_state(state, valid_mask):
    scaled = np.zeros_like(
        state,
        dtype=np.float32,
    )

    for band_index in [0, 1]:
        values = (
            state[band_index]
            - sar_low[band_index]
        ) / (
            sar_high[band_index]
            - sar_low[band_index]
        )

        values = np.clip(
            values,
            0.0,
            1.0,
        )

        scaled[band_index] = np.where(
            valid_mask[band_index],
            values,
            0.0,
        )

    return np.transpose(
        scaled,
        (1, 2, 0),
    ).astype(np.float32)


def load_pre_post(raster_path):
    raster, valid = read_raw_patch(raster_path)

    pre = scale_state(
        raster[0:2],
        valid[0:2],
    )

    post = scale_state(
        raster[2:4],
        valid[2:4],
    )

    valid_land = np.logical_and(
        valid[0:2].all(axis=0),
        valid[2:4].all(axis=0),
    )

    raw_change = np.transpose(
        raster[4:6],
        (1, 2, 0),
    )

    raw_change_valid = np.transpose(
        valid[4:6],
        (1, 2, 0),
    )

    return (
        pre,
        post,
        valid_land,
        raw_change,
        raw_change_valid,
    )


## 6. Build pre-disaster autoencoder datasets

Only pre-event VV/VH is used to train the model. Random flips and 90-degree rotations are applied to both the autoencoder input and target, so the reconstruction task remains physically consistent.


In [7]:
def py_load_pre(raster_path):
    if hasattr(raster_path, "numpy"):
        raster_path = raster_path.numpy().decode("utf-8")
    elif isinstance(raster_path, bytes):
        raster_path = raster_path.decode("utf-8")

    pre, _, _, _, _ = load_pre_post(raster_path)
    return pre.astype(np.float32)


def tf_load_pre(raster_path):
    image = tf.py_function(
        py_load_pre,
        [raster_path],
        Tout=tf.float32,
    )

    image.set_shape(
        [
            PATCH_HEIGHT,
            PATCH_WIDTH,
            2,
        ]
    )

    return image


def augment_pair(image):
    image = tf.image.random_flip_left_right(
        image,
        seed=SEED,
    )

    image = tf.image.random_flip_up_down(
        image,
        seed=SEED + 1,
    )

    k = tf.random.uniform(
        shape=[],
        minval=0,
        maxval=4,
        dtype=tf.int32,
        seed=SEED + 2,
    )

    image = tf.image.rot90(
        image,
        k=k,
    )

    return image, image


def identity_pair(image):
    return image, image


def make_autoencoder_dataset(
    dataframe,
    batch_size,
    training,
):
    dataset = tf.data.Dataset.from_tensor_slices(
        dataframe["raster_path"].astype(str).values
    )

    if training:
        dataset = dataset.shuffle(
            len(dataframe),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    dataset = dataset.map(
        tf_load_pre,
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    dataset = dataset.map(
        augment_pair if training else identity_pair,
        num_parallel_calls=tf.data.AUTOTUNE,
    )

    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


## 7. Convolutional autoencoder

The architecture uses the CNN elements covered in the course: small convolution kernels, ReLU activation, pooling, a compressed latent representation and convolutional decoding.

The output activation is sigmoid because the inputs have been scaled to 0-1. Mean absolute error is used as the reconstruction objective so that unusual post-disaster structure produces a directly interpretable residual.


In [8]:
def build_autoencoder(params):
    kernel_size = int(params["kernel_size"])
    filters_1 = int(params["filters_1"])
    filters_2 = int(params["filters_2"])
    latent_filters = int(params["latent_filters"])
    dropout = float(params["dropout"])
    learning_rate = float(params["learning_rate"])

    inputs = tf.keras.layers.Input(
        shape=(
            PATCH_HEIGHT,
            PATCH_WIDTH,
            2,
        ),
        name="sar_state",
    )

    x = tf.keras.layers.Conv2D(
        filters_1,
        kernel_size,
        padding="same",
        activation="relu",
    )(inputs)

    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = tf.keras.layers.Conv2D(
        filters_2,
        kernel_size,
        padding="same",
        activation="relu",
    )(x)

    x = tf.keras.layers.MaxPooling2D(2)(x)

    if dropout > 0:
        x = tf.keras.layers.Dropout(dropout)(x)

    latent = tf.keras.layers.Conv2D(
        latent_filters,
        kernel_size,
        padding="same",
        activation="relu",
        name="latent_features",
    )(x)

    x = tf.keras.layers.UpSampling2D(2)(latent)

    x = tf.keras.layers.Conv2D(
        filters_2,
        kernel_size,
        padding="same",
        activation="relu",
    )(x)

    x = tf.keras.layers.UpSampling2D(2)(x)

    x = tf.keras.layers.Conv2D(
        filters_1,
        kernel_size,
        padding="same",
        activation="relu",
    )(x)

    outputs = tf.keras.layers.Conv2D(
        2,
        1,
        padding="same",
        activation="sigmoid",
        name="reconstruction",
    )(x)

    autoencoder = tf.keras.Model(
        inputs,
        outputs,
        name="normal_port_autoencoder",
    )

    encoder = tf.keras.Model(
        inputs,
        latent,
        name="normal_port_encoder",
    )

    autoencoder.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss="mae",
    )

    return autoencoder, encoder


## 8. Unsupervised hyperparameter tuning

No damage labels are used. Candidate CNN configurations are compared using reconstruction MAE on **pre-disaster validation ports**. Lower validation MAE means the model has learned the normal Sentinel-1 port distribution more effectively without using the held-out test ports.

The search varies convolution width, latent width, kernel size, dropout, learning rate and batch size. Random search is used to keep the compute cost manageable.


In [9]:
parameter_space = {
    "filters_1": [8, 16, 32],
    "filters_2": [16, 32, 64],
    "latent_filters": [16, 32, 64],
    "kernel_size": [3, 5],
    "dropout": [0.0, 0.10],
    "learning_rate": [1e-3, 3e-4],
    "batch_size": [4, 8, 16],
}

sampled_parameters = list(
    ParameterSampler(
        parameter_space,
        n_iter=MAX_TUNING_TRIALS,
        random_state=SEED,
    )
)

# Keep sensible channel progressions.
sampled_parameters = [
    params
    for params in sampled_parameters
    if params["filters_2"] >= params["filters_1"]
    and params["latent_filters"] >= params["filters_2"] / 2
]

if not sampled_parameters:
    raise RuntimeError("No valid hyperparameter combinations were generated.")

print("Hyperparameter trials:", len(sampled_parameters))


Hyperparameter trials: 9


In [ ]:
tuning_rows = []

for trial_number, params in enumerate(
    sampled_parameters,
    start=1,
):
    print()
    print(
        f"Trial {trial_number}/{len(sampled_parameters)}"
    )
    print(params)

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED + trial_number)

    batch_size = int(params["batch_size"])

    train_ds = make_autoencoder_dataset(
        train_df,
        batch_size=batch_size,
        training=True,
    )

    val_ds = make_autoencoder_dataset(
        val_df,
        batch_size=batch_size,
        training=False,
    )

    model, _ = build_autoencoder(params)

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=3,
            restore_best_weights=True,
        )
    ]

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=TUNING_EPOCHS,
        callbacks=callbacks,
        verbose=0,
    )

    best_val_loss = float(
        np.min(history.history["val_loss"])
    )

    best_epoch = int(
        np.argmin(history.history["val_loss"])
        + 1
    )

    tuning_rows.append(
        {
            **params,
            "best_val_mae": best_val_loss,
            "best_epoch": best_epoch,
            "parameters": int(model.count_params()),
        }
    )

    print(
        "Best validation MAE:",
        f"{best_val_loss:.6f}",
    )

tuning_results = (
    pd.DataFrame(tuning_rows)
    .sort_values(
        ["best_val_mae", "parameters"]
    )
    .reset_index(drop=True)
)

tuning_results.to_csv(
    TUNING_RESULTS_PATH,
    index=False,
)

display(tuning_results)



Trial 1/9
{'learning_rate': 0.0003, 'latent_filters': 64, 'kernel_size': 5, 'filters_2': 32, 'filters_1': 32, 'dropout': 0.1, 'batch_size': 16}
Best validation MAE: 0.131367

Trial 2/9
{'learning_rate': 0.001, 'latent_filters': 64, 'kernel_size': 3, 'filters_2': 16, 'filters_1': 8, 'dropout': 0.0, 'batch_size': 8}
Best validation MAE: 0.076968

Trial 3/9
{'learning_rate': 0.0003, 'latent_filters': 64, 'kernel_size': 3, 'filters_2': 32, 'filters_1': 16, 'dropout': 0.1, 'batch_size': 16}
Best validation MAE: 0.312070

Trial 4/9
{'learning_rate': 0.0003, 'latent_filters': 32, 'kernel_size': 5, 'filters_2': 64, 'filters_1': 32, 'dropout': 0.1, 'batch_size': 8}
Best validation MAE: 0.083500

Trial 5/9
{'learning_rate': 0.001, 'latent_filters': 64, 'kernel_size': 3, 'filters_2': 32, 'filters_1': 8, 'dropout': 0.0, 'batch_size': 16}
Best validation MAE: 0.072181

Trial 6/9
{'learning_rate': 0.0003, 'latent_filters': 64, 'kernel_size': 5, 'filters_2': 32, 'filters_1': 8, 'dropout': 0.1, 'batc

In [ ]:
best_row = tuning_results.iloc[0]

best_params = {
    "filters_1": int(best_row["filters_1"]),
    "filters_2": int(best_row["filters_2"]),
    "latent_filters": int(best_row["latent_filters"]),
    "kernel_size": int(best_row["kernel_size"]),
    "dropout": float(best_row["dropout"]),
    "learning_rate": float(best_row["learning_rate"]),
    "batch_size": int(best_row["batch_size"]),
}

with open(BEST_PARAMS_PATH, "w") as file:
    json.dump(best_params, file, indent=2)

print("Selected hyperparameters:")
print(json.dumps(best_params, indent=2))


## 9. Train the selected normal-port model


In [ ]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

best_batch_size = int(
    best_params["batch_size"]
)

train_ds = make_autoencoder_dataset(
    train_df,
    batch_size=best_batch_size,
    training=True,
)

val_ds = make_autoencoder_dataset(
    val_df,
    batch_size=best_batch_size,
    training=False,
)

autoencoder, encoder = build_autoencoder(
    best_params
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
    ),
]

history = autoencoder.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINAL_EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

encoder.save(ENCODER_PATH)


In [ ]:
history_df = pd.DataFrame(history.history)

fig, ax = plt.subplots(figsize=(7, 4.5))

ax.plot(
    np.arange(1, len(history_df) + 1),
    history_df["loss"],
    label="Training pre-event MAE",
)

ax.plot(
    np.arange(1, len(history_df) + 1),
    history_df["val_loss"],
    label="Validation pre-event MAE",
)

ax.set_xlabel("Epoch")
ax.set_ylabel("Reconstruction MAE")
ax.set_title("Unsupervised normal-port autoencoder training")
ax.legend(frameon=False)
ax.grid(alpha=0.2)

fig.tight_layout()

training_plot_path = (
    UNSUPERVISED_DIR / "autoencoder_training_curve.png"
)

fig.savefig(
    training_plot_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved:", training_plot_path)


## 10. Check generalisation on held-out pre-disaster ports

Because the workflow is unsupervised, this is not a damage accuracy test. It checks whether the normal-port autoencoder can reconstruct pre-disaster imagery from ports that were not used during training.


In [ ]:
def reconstruction_mae_for_dataframe(dataframe):
    rows = []

    for _, row in dataframe.iterrows():
        pre, _, valid_land, _, _ = load_pre_post(
            row["raster_path"]
        )

        reconstruction = autoencoder.predict(
            pre[None, ...],
            verbose=0,
        )[0]

        error = np.mean(
            np.abs(
                reconstruction - pre
            ),
            axis=-1,
        )

        valid_values = error[valid_land]

        rows.append(
            {
                "sample_id": row["sample_id"],
                "source_port_id": row["source_port_id"],
                "pre_reconstruction_mae": float(
                    np.mean(valid_values)
                ),
            }
        )

    return pd.DataFrame(rows)

pre_train_diagnostics = reconstruction_mae_for_dataframe(
    train_df
)

pre_val_diagnostics = reconstruction_mae_for_dataframe(
    val_df
)

pre_test_diagnostics = reconstruction_mae_for_dataframe(
    test_df
)

print(
    "Training pre-event MAE:",
    pre_train_diagnostics[
        "pre_reconstruction_mae"
    ].mean(),
)

print(
    "Validation pre-event MAE:",
    pre_val_diagnostics[
        "pre_reconstruction_mae"
    ].mean(),
)

print(
    "Held-out test pre-event MAE:",
    pre_test_diagnostics[
        "pre_reconstruction_mae"
    ].mean(),
)


## 11. Calibrate the normal reconstruction-error distribution

A pixel-level normal-error threshold is learned from pre-disaster training imagery only. The 99th percentile is used as a conservative reference for reconstruction errors that are unusual under the learned normal-port model.


In [ ]:
normal_pixel_errors = []

for raster_path in train_df["raster_path"]:
    pre, _, valid_land, _, _ = load_pre_post(raster_path)

    pre_reconstruction = autoencoder.predict(
        pre[None, ...],
        verbose=0,
    )[0]

    pre_error = np.mean(
        np.abs(
            pre_reconstruction - pre
        ),
        axis=-1,
    )

    normal_pixel_errors.append(
        pre_error[valid_land]
    )

normal_pixel_errors = np.concatenate(
    normal_pixel_errors
)

NORMAL_PIXEL_P99 = float(
    np.percentile(
        normal_pixel_errors,
        99,
    )
)

NORMAL_PIXEL_MEDIAN = float(
    np.median(normal_pixel_errors)
)

NORMAL_PIXEL_MAD = float(
    np.median(
        np.abs(
            normal_pixel_errors
            - NORMAL_PIXEL_MEDIAN
        )
    )
)

print("Normal error median:", NORMAL_PIXEL_MEDIAN)
print("Normal error MAD:", NORMAL_PIXEL_MAD)
print("Normal error 99th percentile:", NORMAL_PIXEL_P99)


## 12. Extract unsupervised change features

For each patch the notebook calculates:

- pre-event reconstruction MAE;
- post-event reconstruction MAE;
- positive increase in reconstruction error after the disaster;
- 95th percentile of that reconstruction-error increase;
- fraction of land pixels whose post-event error exceeds the normal 99th percentile and is worse than the pre-event error;
- distance between CNN latent feature maps for pre and post imagery;
- mean absolute VV/VH dB change from the original preprocessing pipeline.

The first five features come directly from the CNN autoencoder. The raw SAR change term provides a physically interpretable reference to the observed radar change.


In [ ]:
def extract_change_features(row, save_map=False):
    (
        pre,
        post,
        valid_land,
        raw_change,
        raw_change_valid,
    ) = load_pre_post(row["raster_path"])

    valid_land_count = int(valid_land.sum())
    valid_land_fraction = float(valid_land.mean())

    pre_reconstruction = autoencoder.predict(
        pre[None, ...],
        verbose=0,
    )[0]

    post_reconstruction = autoencoder.predict(
        post[None, ...],
        verbose=0,
    )[0]

    pre_error = np.mean(
        np.abs(pre_reconstruction - pre),
        axis=-1,
    )

    post_error = np.mean(
        np.abs(post_reconstruction - post),
        axis=-1,
    )

    error_increase = np.maximum(
        post_error - pre_error,
        0.0,
    )

    anomaly_mask = (
        valid_land
        & (post_error > NORMAL_PIXEL_P99)
        & (post_error > pre_error)
    )

    # Some coastal grid cells can contain no valid land after water masking.
    # These patches cannot support land-based change statistics, so return NaN
    # for the clustering features and let the clustering step exclude them.
    if valid_land_count == 0:
        features = {
            "sample_id": row["sample_id"],
            "source_port_id": row["source_port_id"],
            "port_key": row["port_key"],
            "sample_type": row["sample_type"],
            "port_name": row["port_name"],
            "country": row["country"],
            "disaster": row["disaster"],
            "storm_sid": row.get("storm_sid", np.nan),
            "storm_name": row.get("storm_name", np.nan),
            "exposure_class": row.get("exposure_class", np.nan),
            "control_for": row.get("control_for", ""),
            "north_offset_m": row["north_offset_m"],
            "east_offset_m": row["east_offset_m"],
            "pre_reconstruction_mae": np.nan,
            "post_reconstruction_mae": np.nan,
            "reconstruction_change_mean": np.nan,
            "reconstruction_change_p95": np.nan,
            "anomaly_pixel_fraction": np.nan,
            "latent_change": np.nan,
            "raw_change_mean_abs_db": np.nan,
            "valid_land_fraction": valid_land_fraction,
            "quality_status": "no_valid_land",
        }

        if save_map:
            return features, error_increase, anomaly_mask

        return features

    valid_error_increase = error_increase[valid_land]

    pre_latent = encoder.predict(
        pre[None, ...],
        verbose=0,
    )[0]

    post_latent = encoder.predict(
        post[None, ...],
        verbose=0,
    )[0]

    latent_distance = float(
        np.mean(np.abs(post_latent - pre_latent))
    )

    raw_valid = (
        raw_change_valid.all(axis=-1)
        & valid_land
    )

    raw_change_abs = np.mean(
        np.abs(raw_change),
        axis=-1,
    )

    raw_change_mean_abs = (
        float(np.mean(raw_change_abs[raw_valid]))
        if np.any(raw_valid)
        else np.nan
    )

    features = {
        "sample_id": row["sample_id"],
        "source_port_id": row["source_port_id"],
        "port_key": row["port_key"],
        "sample_type": row["sample_type"],
        "port_name": row["port_name"],
        "country": row["country"],
        "disaster": row["disaster"],
        "storm_sid": row.get("storm_sid", np.nan),
        "storm_name": row.get("storm_name", np.nan),
        "exposure_class": row.get("exposure_class", np.nan),
        "control_for": row.get("control_for", ""),
        "north_offset_m": row["north_offset_m"],
        "east_offset_m": row["east_offset_m"],
        "pre_reconstruction_mae": float(
            np.mean(pre_error[valid_land])
        ),
        "post_reconstruction_mae": float(
            np.mean(post_error[valid_land])
        ),
        "reconstruction_change_mean": float(
            np.mean(valid_error_increase)
        ),
        "reconstruction_change_p95": float(
            np.percentile(valid_error_increase, 95)
        ),
        "anomaly_pixel_fraction": float(
            anomaly_mask.sum() / valid_land_count
        ),
        "latent_change": latent_distance,
        "raw_change_mean_abs_db": raw_change_mean_abs,
        "valid_land_fraction": valid_land_fraction,
        "quality_status": "valid",
    }

    if save_map:
        return features, error_increase, anomaly_mask

    return features


In [ ]:
feature_rows = []

for _, row in manifest.iterrows():
    feature_rows.append(
        extract_change_features(row)
    )

change_features = pd.DataFrame(feature_rows)

no_land_count = int(
    (change_features["quality_status"] == "no_valid_land").sum()
)

print("Patches with no valid land and excluded from clustering:", no_land_count)

display(change_features.head())

change_features.to_csv(
    UNSUPERVISED_DIR / "cnn_change_features.csv",
    index=False,
)


## 13. Unsupervised clustering of change signatures

K-means is fitted only on development patches with enough valid land to support stable statistics. This prevents almost-empty shoreline patches from defining the high-change cluster.

Because the current dataset is small, a second unsupervised screening check is also used. A patch is treated as high-change when either K-means assigns it to the higher-change cluster, or at least two of three interpretable change indicators exceed the 80th percentile of the quality-controlled development patches. No damage labels are used to calculate these thresholds.


In [ ]:
CLUSTER_FEATURES = [
    "reconstruction_change_mean",
    "reconstruction_change_p95",
    "anomaly_pixel_fraction",
    "latent_change",
    "raw_change_mean_abs_db",
]

clustering_df = change_features.dropna(
    subset=CLUSTER_FEATURES
).copy()

# The current dataset contains some shoreline patches with almost no valid land.
# Do not let these near-empty patches define the K-means cluster centres.
MIN_CLUSTER_LAND_FRACTION = 0.01

development_ports = train_ports | val_ports

development_mask = clustering_df[
    "port_key"
].isin(development_ports)

cluster_fit_mask = (
    development_mask
    & (
        clustering_df["valid_land_fraction"]
        >= MIN_CLUSTER_LAND_FRACTION
    )
)

if cluster_fit_mask.sum() < 4:
    raise ValueError(
        "Not enough quality-controlled development patches are available "
        "for clustering."
    )

scaler = StandardScaler()

development_scaled = scaler.fit_transform(
    clustering_df.loc[
        cluster_fit_mask,
        CLUSTER_FEATURES,
    ]
)

kmeans = KMeans(
    n_clusters=2,
    random_state=SEED,
    n_init=50,
)

kmeans.fit(development_scaled)

all_scaled = scaler.transform(
    clustering_df[CLUSTER_FEATURES]
)

clustering_df["change_cluster"] = kmeans.predict(
    all_scaled
)

cluster_summary = (
    clustering_df.loc[cluster_fit_mask]
    .groupby("change_cluster")[CLUSTER_FEATURES]
    .mean()
)

display(cluster_summary)

# Use all five change measures to identify the more severe cluster.
cluster_severity = cluster_summary.rank(pct=True).sum(axis=1)

HIGH_CHANGE_CLUSTER = int(
    cluster_severity.idxmax()
)

# Additional label-free screening thresholds.
# These are calculated from the development data only, not from known damage labels.
SCREEN_FEATURES = [
    "reconstruction_change_p95",
    "anomaly_pixel_fraction",
    "raw_change_mean_abs_db",
]

SCREEN_QUANTILE = 0.80

screen_thresholds = clustering_df.loc[
    cluster_fit_mask,
    SCREEN_FEATURES,
].quantile(SCREEN_QUANTILE)

print("Screening thresholds:")
display(screen_thresholds.to_frame("threshold"))

evidence_matrix = pd.DataFrame(
    {
        feature: (
            clustering_df[feature]
            > screen_thresholds[feature]
        )
        for feature in SCREEN_FEATURES
    },
    index=clustering_df.index,
)

clustering_df["change_evidence_count"] = (
    evidence_matrix.sum(axis=1)
)

clustering_df["kmeans_high_change"] = (
    clustering_df["change_cluster"]
    == HIGH_CHANGE_CLUSTER
)

clustering_df["multi_indicator_high_change"] = (
    (
        clustering_df["valid_land_fraction"]
        >= MIN_CLUSTER_LAND_FRACTION
    )
    & (clustering_df["change_evidence_count"] >= 2)
)

clustering_df["potential_damage_anomaly"] = (
    clustering_df["kmeans_high_change"]
    | clustering_df["multi_indicator_high_change"]
)

development_labels = kmeans.labels_

if (
    len(development_scaled) >= 3
    and len(np.unique(development_labels)) == 2
):
    silhouette = silhouette_score(
        development_scaled,
        development_labels,
    )
else:
    silhouette = np.nan

print("Higher-change cluster:", HIGH_CHANGE_CLUSTER)
print("Development K-means silhouette score:", silhouette)
print(
    "Development patches used to fit K-means:",
    int(cluster_fit_mask.sum()),
)


In [ ]:
# Continuous anomaly index based on the standardized change representation.
cluster_centres = kmeans.cluster_centers_

low_change_cluster = 1 - HIGH_CHANGE_CLUSTER

high_centre = cluster_centres[HIGH_CHANGE_CLUSTER]
low_centre = cluster_centres[low_change_cluster]

high_distance = np.linalg.norm(
    all_scaled - high_centre,
    axis=1,
)

low_distance = np.linalg.norm(
    all_scaled - low_centre,
    axis=1,
)

# Positive values are closer to the higher-change cluster.
clustering_df[
    "anomaly_index"
] = low_distance - high_distance

result_columns = [
    "sample_id",
    "source_port_id",
    "port_key",
    "sample_type",
    "port_name",
    "country",
    "disaster",
    "storm_sid",
    "storm_name",
    "exposure_class",
    "control_for",
    "north_offset_m",
    "east_offset_m",
    *CLUSTER_FEATURES,
    "change_cluster",
    "kmeans_high_change",
    "change_evidence_count",
    "multi_indicator_high_change",
    "potential_damage_anomaly",
    "anomaly_index",
    "valid_land_fraction",
    "quality_status",
]

patch_results = clustering_df[result_columns].copy()

patch_results.to_csv(
    PATCH_RESULTS_PATH,
    index=False,
)

display(
    patch_results.sort_values(
        "anomaly_index",
        ascending=False,
    ).head(15)
)


## 14. Port-level anomaly decision

Patch-level evidence is combined across the 3 × 3 grid. A port is flagged when at least two quality-controlled patches show high change, or when the central patch itself is high-change. This spatial-consensus rule reduces reliance on a single isolated patch while remaining sensitive to damage that affects only part of a port.


In [ ]:
# Combine patch-level results into one result per port-event.
# source_port_id identifies the whole 3 x 3 assessment; sample_id identifies one patch.

central_flags = (
    patch_results[
        (patch_results["north_offset_m"] == 0)
        & (patch_results["east_offset_m"] == 0)
    ]
    [["source_port_id", "potential_damage_anomaly"]]
    .rename(
        columns={
            "potential_damage_anomaly": "central_patch_anomaly"
        }
    )
    .drop_duplicates(subset=["source_port_id"])
)

port_group_columns = [
    "source_port_id",
    "port_key",
    "sample_type",
    "port_name",
    "country",
    "disaster",
    "storm_name",
    "exposure_class",
    "control_for",
]

port_results = (
    patch_results
    .groupby(
        port_group_columns,
        as_index=False,
        dropna=False,
    )
    .agg(
        mean_anomaly_index=("anomaly_index", "mean"),
        max_anomaly_index=("anomaly_index", "max"),
        anomalous_patch_fraction=("potential_damage_anomaly", "mean"),
        anomalous_patch_count=("potential_damage_anomaly", "sum"),
        number_valid_patches=("potential_damage_anomaly", "size"),
        mean_valid_land_fraction=("valid_land_fraction", "mean"),
    )
)

port_results = port_results.merge(
    central_flags,
    on="source_port_id",
    how="left",
)

port_results["central_patch_anomaly"] = (
    port_results["central_patch_anomaly"]
    .fillna(False)
    .astype(bool)
)

# Spatial-consensus rule:
# - at least two high-change patches anywhere in the grid, OR
# - a high-change central patch.
MIN_HIGH_CHANGE_PATCHES = 2

port_results["potential_surface_damage"] = (
    (port_results["anomalous_patch_count"] >= MIN_HIGH_CHANGE_PATCHES)
    | port_results["central_patch_anomaly"]
)

port_results["surface_status"] = np.where(
    port_results["potential_surface_damage"],
    "potential_surface_damage",
    "no_large_anomaly_detected",
)

port_results.to_csv(
    PORT_RESULTS_PATH,
    index=False,
)

display(
    port_results.sort_values(
        ["potential_surface_damage", "max_anomaly_index"],
        ascending=[False, False],
    )
)


In [ ]:
# Diagnostic: show exactly why Luganville is or is not being flagged.
luganville_patch_check = patch_results[
    (patch_results["port_name"].str.upper() == "LUGANVILLE")
    & (patch_results["storm_name"].astype(str).str.lower() == "harold")
][
    [
        "sample_id",
        "valid_land_fraction",
        "reconstruction_change_p95",
        "anomaly_pixel_fraction",
        "raw_change_mean_abs_db",
        "kmeans_high_change",
        "change_evidence_count",
        "multi_indicator_high_change",
        "potential_damage_anomaly",
        "anomaly_index",
    ]
]

display(luganville_patch_check)

luganville_port_check = port_results[
    (port_results["port_name"].str.upper() == "LUGANVILLE")
    & (port_results["storm_name"].astype(str).str.lower() == "harold")
]

display(luganville_port_check)


## 15. Compare cyclone events with quiet temporal controls

These comparisons do not create damage labels. They check whether cyclone-period samples produce stronger anomaly scores than quiet periods from the same physical ports.


In [ ]:
# Overall comparison by sample type, now using one row per port-event.
sample_type_summary = (
    port_results
    .groupby("sample_type", dropna=False)
    .agg(
        number_samples=("source_port_id", "size"),
        mean_anomaly_index=("mean_anomaly_index", "mean"),
        mean_max_anomaly=("max_anomaly_index", "mean"),
        mean_anomalous_patch_fraction=("anomalous_patch_fraction", "mean"),
        flagged_fraction=("potential_surface_damage", "mean"),
    )
    .reset_index()
)

display(sample_type_summary)

events = port_results[
    port_results["sample_type"] == "event"
].copy()

controls = port_results[
    port_results["sample_type"] == "control_temporal"
].copy()

# Normalise IDs in case an older manifest stored a patch suffix in control_for.
events["event_link_id"] = (
    events["source_port_id"]
    .astype(str)
    .str.replace(r"_r\d+_c\d+$", "", regex=True)
)

controls["control_for_link_id"] = (
    controls["control_for"]
    .astype(str)
    .str.replace(r"_r\d+_c\d+$", "", regex=True)
)

control_lookup = controls[
    [
        "source_port_id",
        "control_for_link_id",
        "mean_anomaly_index",
        "max_anomaly_index",
        "anomalous_patch_fraction",
        "potential_surface_damage",
    ]
].rename(
    columns={
        "source_port_id": "control_source_port_id",
        "mean_anomaly_index": "control_mean_anomaly",
        "max_anomaly_index": "control_max_anomaly",
        "anomalous_patch_fraction": "control_anomalous_patch_fraction",
        "potential_surface_damage": "control_flagged",
    }
)

event_control_comparison = events.merge(
    control_lookup,
    left_on="event_link_id",
    right_on="control_for_link_id",
    how="left",
)

event_control_comparison["mean_anomaly_difference"] = (
    event_control_comparison["mean_anomaly_index"]
    - event_control_comparison["control_mean_anomaly"]
)

event_control_comparison["max_anomaly_difference"] = (
    event_control_comparison["max_anomaly_index"]
    - event_control_comparison["control_max_anomaly"]
)

comparison_columns = [
    "port_name",
    "storm_name",
    "exposure_class",
    "mean_anomaly_index",
    "control_mean_anomaly",
    "mean_anomaly_difference",
    "max_anomaly_index",
    "control_max_anomaly",
    "max_anomaly_difference",
    "potential_surface_damage",
    "control_flagged",
]

display(
    event_control_comparison[comparison_columns]
    .sort_values("max_anomaly_difference", ascending=False)
)

event_control_comparison.to_csv(
    UNSUPERVISED_DIR / "event_control_comparison.csv",
    index=False,
)


## 16. Check the independently verified Luganville case

Luganville–Harold is retained as the independently verified case because the post-disaster assessment reports damage to the port. The table below compares the model's port-level prediction with that evidence.

**Important:** one verified case is useful as a proof-of-concept check, but it is not enough to calculate a meaningful accuracy, precision, recall or F1 score.


In [ ]:
verified_cases = pd.DataFrame(
    [
        {
            "port_name": "LUGANVILLE",
            "storm_name": "Harold",
            "verified_surface_damage": True,
        }
    ]
)

verification = port_results.merge(
    verified_cases,
    on=["port_name", "storm_name"],
    how="inner",
)

verification["classification_correct"] = (
    verification["potential_surface_damage"]
    == verification["verified_surface_damage"]
)

display(
    verification[
        [
            "port_name",
            "storm_name",
            "anomalous_patch_count",
            "number_valid_patches",
            "potential_surface_damage",
            "verified_surface_damage",
            "classification_correct",
        ]
    ]
)

if verification.empty:
    print(
        "Luganville–Harold is not present in the current manifest. "
        "Rerun the data-preparation notebook with this event included."
    )

verification.to_csv(
    UNSUPERVISED_DIR / "verified_case_check.csv",
    index=False,
)


## 17. Visualise anomaly maps and 3 x 3 port grids


In [ ]:
skipped_maps = 0

for _, row in manifest.iterrows():
    features, error_map, anomaly_mask = extract_change_features(
        row,
        save_map=True,
    )

    pre, post, valid_land, _, _ = load_pre_post(
        row["raster_path"]
    )

    if not np.any(valid_land):
        skipped_maps += 1
        continue

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(14, 3.5),
    )

    axes[0].imshow(pre[:, :, 0], cmap="gray", vmin=0, vmax=1)
    axes[0].set_title("Pre-event VV")

    axes[1].imshow(post[:, :, 0], cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Post-event VV")

    change_plot = axes[2].imshow(
        np.where(valid_land, error_map, np.nan),
        cmap="magma",
    )
    axes[2].set_title("CNN error increase")
    fig.colorbar(
        change_plot,
        ax=axes[2],
        fraction=0.046,
        pad=0.04,
    )

    axes[3].imshow(
        np.where(valid_land, anomaly_mask, np.nan),
        cmap="Reds",
        vmin=0,
        vmax=1,
    )
    axes[3].set_title("High-error pixels")

    for ax in axes:
        ax.axis("off")

    fig.suptitle(row["sample_id"])
    fig.tight_layout()

    output_path = (
        UNSUPERVISED_DIR
        / f"{row['sample_id']}_change_map.png"
    )

    fig.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight",
    )

    plt.close(fig)

print("Patch anomaly maps saved to:", UNSUPERVISED_DIR)
print("Maps skipped because no valid land was present:", skipped_maps)


In [ ]:
for port_id, frame in patch_results.groupby(
    "source_port_id"
):
    grid = (
        frame
        .pivot_table(
            index="north_offset_m",
            columns="east_offset_m",
            values="anomaly_index",
            aggfunc="mean",
        )
        .sort_index(ascending=False)
    )

    fig, ax = plt.subplots(figsize=(5.5, 5))

    image = ax.imshow(
        grid.values,
        cmap="RdYlBu_r",
    )

    for row_index in range(grid.shape[0]):
        for column_index in range(grid.shape[1]):
            value = grid.iloc[row_index, column_index]
            if np.isfinite(value):
                ax.text(
                    column_index,
                    row_index,
                    f"{value:.2f}",
                    ha="center",
                    va="center",
                )

    ax.set_xticks(range(grid.shape[1]))
    ax.set_xticklabels([int(x) for x in grid.columns])
    ax.set_yticks(range(grid.shape[0]))
    ax.set_yticklabels([int(y) for y in grid.index])
    ax.set_xlabel("East offset from port centre (m)")
    ax.set_ylabel("North offset from port centre (m)")
    ax.set_title(f"Unsupervised change index: {port_id}")

    colourbar = fig.colorbar(image, ax=ax)
    colourbar.set_label("Anomaly index")

    fig.tight_layout()

    grid_path = (
        UNSUPERVISED_DIR
        / f"{port_id}_anomaly_grid.png"
    )

    fig.savefig(
        grid_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()


## 17. Save the clustering configuration

The autoencoder and encoder are stored by Keras. The scaler and K-means values are saved as JSON so the same anomaly representation can be applied to new ports.


In [ ]:
cluster_config = {
    "cluster_features": CLUSTER_FEATURES,
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "kmeans_centres": kmeans.cluster_centers_.tolist(),
    "high_change_cluster": int(HIGH_CHANGE_CLUSTER),
    "normal_pixel_p99": float(NORMAL_PIXEL_P99),
    "normal_pixel_median": float(NORMAL_PIXEL_MEDIAN),
    "normal_pixel_mad": float(NORMAL_PIXEL_MAD),
    "silhouette_score": (
        float(silhouette)
        if np.isfinite(silhouette)
        else None
    ),
    "split_group": "port_key",
    "training_ports": sorted(train_ports),
    "validation_ports": sorted(val_ports),
    "test_ports": sorted(test_ports),
}

with open(CLUSTER_PATH, "w") as file:
    json.dump(
        cluster_config,
        file,
        indent=2,
    )

print("Saved autoencoder:", MODEL_PATH)
print("Saved encoder:", ENCODER_PATH)
print("Saved clustering configuration:", CLUSTER_PATH)


## 18. Final project output

The unsupervised CNN does not prove that a structure is damaged in an engineering sense. It identifies non-trivial post-disaster anomalies on and around the port that are consistent with potential surface damage.

The final operational decision should combine this surface-change result with the independent bathymetry/access model:

port usable for vessel class = no large surface anomaly detected AND sufficient water depth

Cases with high anomaly scores should be prioritised for human review or higher-resolution imagery rather than automatically treated as definitively unusable.


In [ ]:
port_results = pd.read_csv(
    UNSUPERVISED_DIR / "unsupervised_port_results.csv"
)

display(port_results)